# Day 17 (Part 2): Preparing Data for Training

Part 1 ([`Notebook.ipynb`](Notebook.ipynb)) built `CustomerPredictor`, a small network with 3 inputs and 1 output. Before it can learn, it needs data. This notebook prepares that data step by step, until it is ready to be fed into a training loop.

![Part 2 roadmap: create the dataset, split it, scale it, convert it to tensors, cut it into mini-batches, then train](images/data_pipeline.svg)

### How to use this notebook
- Run the cells **from top to bottom**. Each step uses the variables made by the step before it (`X`, `X_train`, `X_train_scaled`, ...).
- Each section follows the same pattern: **the idea → the code → what the output means**.
- The diagrams are in the `images/` folder.

### Contents

| Section | What you will learn |
|---|---|
| 1. Importing the Libraries | Which library does which job |
| 2. Creating the Dataset | A practice dataset of 1000 customers with `make_classification` |
| 3. Splitting into Training, Validation and Test Sets | Why the data is split three ways |
| 4. Feature Scaling with `StandardScaler` | Scaling, and `fit_transform` vs `transform` |
| 5. Converting the Data to PyTorch Tensors | `float32` and the `(n, 1)` label shape |
| 6. Mini-batches with `TensorDataset` and `DataLoader` | Batches, iterations, epochs and shuffling |
| 7. Creating the Model | The same `CustomerPredictor` as Part 1 |
| 8. The Loss Function: Binary Cross-Entropy (BCE) | How the model's mistakes become one number |

## 1. Importing the Libraries

All the tools for this notebook are imported once, at the top. Each one has one job:

| Import | Job | Used in |
|---|---|---|
| `torch`, `torch.nn` | Tensors, layers and loss functions | Sections 5, 7, 8 |
| `torch.optim` | Optimisers that update the weights (SGD, Adam) | Training |
| `TensorDataset`, `DataLoader` | Cut the data into mini-batches | Section 6 |
| `make_classification` | Creates a practice dataset | Section 2 |
| `train_test_split` | Splits the data into training, validation and test sets | Section 3 |
| `StandardScaler` | Puts every feature on the same scale | Section 4 |
| `accuracy_score`, `classification_report`, `confusion_matrix` | Measure how good the predictions are | Evaluation |
| `matplotlib.pyplot` | Draws charts, such as the training loss | Training |

> `torch.optim`, the metrics and `plt` are not used yet. They are needed later, when the model is trained and evaluated.

In [2]:
# PyTorch: tensors, layers, optimisers and data loading
import torch
import torch.nn as nn  # layers (nn.Linear) and loss functions
import torch.optim as optim  # optimisers that update the weights (e.g. SGD, Adam)
from torch.utils.data import (
    DataLoader,
    TensorDataset,
)  # feed the data to the model in mini-batches

# scikit-learn: create, split, scale and score the data
from sklearn.datasets import (
    make_classification,
)  # generates a fake classification dataset
from sklearn.model_selection import (
    train_test_split,
)  # splits the data into training and test sets
from sklearn.preprocessing import (
    StandardScaler,
)  # rescales each feature to mean 0, standard deviation 1

# measure how good the predictions are
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Matplotlib: draw charts (e.g. the training loss)
import matplotlib.pyplot as plt

## 2. Creating the Dataset

Instead of loading a real file, `make_classification` from scikit-learn **generates** a practice dataset. Here it simulates 1000 customers:

| Variable | Shape | What it holds |
|---|---|---|
| `X` | `(1000, 3)` | The **features**: 3 numbers that describe each customer, one row per customer |
| `y` | `(1000,)` | The **labels**: the answer to predict for each customer, `0` or `1` (for example, "will not buy" or "will buy") |

- `n_features=3` matches the first layer of `CustomerPredictor` from Part 1, `nn.Linear(3, 8)`.
- The two classes are almost balanced: 501 zeros and 499 ones.
- `random_state=42` makes scikit-learn create the same data on every run, so your numbers match these notes.

In a real project, this step is usually loading a file, for example with `pd.read_csv(...)`. The rest of the pipeline stays the same.

In [3]:
# Fix PyTorch's random seed so the model starts with the same random weights on every run
torch.manual_seed(42)

# Create a dataset that simulates 1000 customers
X, y = make_classification(
    n_samples=1000,  # 1000 rows: one row per customer
    n_features=3,  # 3 columns per customer, matching the model's 3 inputs
    n_informative=3,  # all 3 features actually help to predict the label
    n_redundant=0,  # no feature is just a mix of the other features
    random_state=42,  # scikit-learn's own seed, so the same data is created every run
)

# X holds the features: one row per customer, one column per feature -> (1000, 3)
# y holds the labels: one 0 or 1 per customer -> (1000,)
print("X shape:", X.shape)
print(X[:5])  # show the first five rows of the features
print("y shape:", y.shape)

X shape: (1000, 3)
[[ 0.52170639  0.72685697  2.50410248]
 [-1.01971039 -0.74986039 -0.71346315]
 [-1.89222477 -1.03217141  1.54176061]
 [ 0.62580061 -0.86628346 -0.35092274]
 [ 1.03461232  1.64138782  3.18254045]]
y shape: (1000,)


## 3. Splitting into Training, Validation and Test Sets

A model must be checked on customers it has **never seen**. Otherwise we cannot tell whether it learned real patterns or just memorised the training rows. So the 1000 rows are split into three sets:

```
X, y (1000 rows)
 ├── 70% ──► X_train, y_train  (700)    the model learns from these
 └── 30% ──► X_temp,  y_temp   (300)
              ├── 50% ──► X_val,  y_val   (150)
              └── 50% ──► X_test, y_test  (150)
```

| Set | Rows | Share | Job |
|---|---|---|---|
| Training | 700 | 70% | The model learns from these: its weights are updated on them. |
| Validation | 150 | 15% | Checked during training, to tune settings (such as the number of epochs) and to spot overfitting. |
| Test | 150 | 15% | Used **once**, at the end, for the final honest score. |

- **Why two calls?** `train_test_split` only splits data into two parts. The first call keeps 70% for training and puts 30% in `X_temp`. The second call cuts `X_temp` in half: 15% for validation and 15% for test.
- **`stratify=y`** keeps the same 0/1 ratio in every set: training has 351 zeros and 349 ones, and validation and test have 75 of each.
- **`random_state=42`** gives the same split on every run.

In [4]:
# Split into training, validation and test sets (Train: 70%, Val: 15%, Test: 15%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,  # keep the same 0/1 ratio in each set
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp,
)


## 4. Feature Scaling with `StandardScaler`

Features often have very different ranges. In real data, age might be 18–80 while yearly spend might be 0–200,000. A feature with big numbers would dominate every weighted sum `x @ Wᵀ + b`, and training would be slow and unstable.

`StandardScaler` rescales **each column** to a mean of 0 and a standard deviation of 1:

$$z = \frac{x - \text{mean}}{\text{std}}$$

After scaling, no feature dominates just because its numbers are bigger. Our generated features are already close to this scale, but scaling is a standard step, and real data almost always needs it.

4.1 explains why the scaler is fitted on the training set only.

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)

### 4.1 ML Guide: `fit_transform` vs `transform`

![StandardScaler: fit once on the training set, transform every set](images/scaler_fit_transform.svg)

| Call | Used on | What it does |
|---|---|---|
| `scaler.fit_transform(X_train)` | Training set | **Learns** each column's mean and std, **then** rescales it: `(x − mean) / std` |
| `scaler.transform(X_val)` | Validation set | Rescales with the **training** mean and std. Learns nothing. |
| `scaler.transform(X_test)` | Test set | Same as above |

**Why not use `fit_transform` on all three?**

1. **Data leakage.** The validation and test sets stand in for new customers the model has never seen. If the scaler learns from them, their information leaks into the pipeline, and the scores measured on them look better than the model really is.
2. **One rule for every set.** The model learns from numbers scaled by the training rule. If each set had its own mean and std, the same raw value would become a different scaled number in each set.
3. **Real-world use.** After deployment, customers arrive one at a time. A single row has no useful mean or std, so it can only be scaled with the saved training numbers: `scaler.transform(new_customer)`.

**What you will see:** the scaled training set has a mean of exactly 0 and a std of exactly 1 per column. The validation and test sets are close but not exact (the test set's means are about `-0.10, 0.16, -0.09`). That is expected, because they were scaled with the training numbers, not their own.

> **Rule of thumb:** split first, then `fit` only on the training data and `transform` everything else. The same rule applies to every preprocessing step that learns from data, such as encoders, imputers and PCA.

## 5. Converting the Data to PyTorch Tensors

`train_test_split` and `StandardScaler` return **NumPy arrays**, but a PyTorch model only works with **tensors**. The cell below converts all six arrays: features (`X`) and labels (`y`) for the training, validation and test sets.

In [6]:
# Convert the NumPy arrays to PyTorch tensors, because a PyTorch model only accepts tensors.
# dtype=torch.float32: the model's weights are float32, so the inputs must be float32 too.
# .view(-1, 1) reshapes the labels from (n_samples,) to (n_samples, 1),
# the same shape as the model's output. -1 means "work out this size from the data".

# Training set: used to update the weights
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

# Validation set: used to check the model during training
X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# Test set: used once, at the end, to measure the final model
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("X_train_t shape:", X_train_t.shape)
print("y_train_t shape:", y_train_t.shape)
print("X_val_t shape:", X_val_t.shape)
print("y_val_t shape:", y_val_t.shape)
print("X_test_t shape:", X_test_t.shape)
print("y_test_t shape:", y_test_t.shape)

X_train_t shape: torch.Size([700, 3])
y_train_t shape: torch.Size([700, 1])
X_val_t shape: torch.Size([150, 3])
y_val_t shape: torch.Size([150, 1])
X_test_t shape: torch.Size([150, 3])
y_test_t shape: torch.Size([150, 1])


### 5.1 What each conversion step does

![Converting NumPy arrays to PyTorch tensors](images/tensor_conversion.svg)

Each part of the conversion prevents a real error. These are the messages PyTorch gives when a step is skipped:

| Step | Why it is needed | Error without it |
|---|---|---|
| `torch.tensor(...)` | PyTorch layers only accept tensors | `TypeError: linear(): argument 'input' (position 1) must be Tensor, not numpy.ndarray` |
| `dtype=torch.float32` on `X` | NumPy gives **float64**, but the weights of `nn.Linear` are **float32**. Both sides of a matrix multiplication must have the same type. | `RuntimeError: mat1 and mat2 must have the same dtype, but got Double and Float` |
| `dtype=torch.float32` on `y` | `make_classification` gives **int64** labels (0 and 1). The loss for a yes/no prediction (`nn.BCEWithLogitsLoss`) needs float labels. | `RuntimeError: result type Float can't be cast to the desired output type Long` |
| `.view(-1, 1)` on `y` | The model returns one score per customer, shape `(batch, 1)`. The loss compares each score with its label, so the labels need the same shape. | `ValueError: Target size (torch.Size([700])) must be the same as input size (torch.Size([700, 1]))` |

**What `-1` means in `.view(-1, 1)`:** "work this size out for me". The `1` fixes the number of columns, so PyTorch computes the rows: 700 values ÷ 1 column = **700 rows**. The same line works for the 150-row validation and test sets without changing the number.

### 5.2 The resulting tensors

| Tensor | Shape | Meaning |
|---|---|---|
| `X_train_t` | `[700, 3]` | 700 customers × 3 features, used to update the weights |
| `y_train_t` | `[700, 1]` | one label (0. or 1.) per training customer |
| `X_val_t`, `y_val_t` | `[150, 3]`, `[150, 1]` | checks the model during training |
| `X_test_t`, `y_test_t` | `[150, 3]`, `[150, 1]` | measures the final model, once, at the end |

## 6. Mini-batches with `TensorDataset` and `DataLoader`

The model does not see all 700 training rows at once. It learns from small groups of rows called **mini-batches**. Two PyTorch tools prepare them: `TensorDataset` pairs each customer with their label, and `DataLoader` cuts those pairs into batches.

In [7]:
# Data loaders: feed the training data to the model in mini-batches

# TensorDataset pairs each row of features with its label,
# so train_dataset[i] gives (X_train_t[i], y_train_t[i])
train_dataset = TensorDataset(X_train_t, y_train_t)

# DataLoader cuts the 700 training rows into mini-batches of 32 rows
# and hands them to the training loop one batch at a time.
# shuffle=True: the rows are shuffled again at the start of every epoch,
# so the batches are different each time.
#
# Why 32? Batch sizes are usually powers of 2: 16, 32, 64, 128, 256 ...
#   - GPUs work in groups of threads whose sizes are powers of 2 (an NVIDIA GPU
#     runs threads in groups of 32), so a batch of that size divides evenly
#     between them and no thread sits idle.
#   - Memory is also laid out in power-of-2 blocks, so the data lines up neatly
#     and less memory and work are wasted.
#   - It is a common convention, so results are easy to compare with others.
# It is a habit, not a rule: any size works (e.g. batch_size=50). On a CPU, like
# here, the speed difference is usually too small to notice.
# Choosing the size: smaller batches (16) -> more updates per epoch, but noisier;
# larger batches (128) -> smoother updates, but more memory and fewer updates.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Rows in the dataset:", len(train_dataset))
print("Batches per epoch:", len(train_loader))  # 700 / 32 = 21.9 -> 22 batches

# Look at the first mini-batch only (list(train_loader) would print all 22 batches)
X_batch, y_batch = next(iter(train_loader))
print("X_batch shape:", X_batch.shape)  # 32 customers x 3 features
print("y_batch shape:", y_batch.shape)  # 32 labels, one per customer

Rows in the dataset: 700
Batches per epoch: 22
X_batch shape: torch.Size([32, 3])
y_batch shape: torch.Size([32, 1])


### 6.1 How the batches are made

![TensorDataset and DataLoader mini-batches](images/dataloader_batches.svg)

| Code | What it does |
|---|---|
| `TensorDataset(X_train_t, y_train_t)` | Pairs row `i` of the features with label `i`. `train_dataset[0]` gives `(tensor([ 1.4325, -1.5463,  1.2566]), tensor([1.]))`: one customer and their label. |
| `DataLoader(..., batch_size=32)` | Cuts the 700 rows into batches of 32. 700 ÷ 32 = 21.9, so there are **22 batches**: 21 full batches of 32 rows and one last batch with the 28 rows left over. |
| `shuffle=True` | Mixes the rows at the start of every epoch, so each batch holds different customers each time. |
| `next(iter(train_loader))` | Takes just the first batch, so you can check its shape. `print(list(train_loader))` would print all 22 batches (700 rows). |

### 6.2 Batch, iteration and epoch

| Word | Meaning in this notebook |
|---|---|
| **Batch** | 32 rows that go through the model together |
| **Iteration** | one batch in → loss → one weight update |
| **Epoch** | one full pass over all 700 rows = **22 iterations** |

### 6.3 Why mini-batches instead of all 700 rows at once?

1. **More learning per epoch:** the weights are updated 22 times per epoch instead of once.
2. **Less memory:** only 32 rows are processed at a time. This matters for real datasets that are too big to fit in memory at once.
3. **Useful randomness:** each batch gives a slightly different estimate of the error. This small "noise" can help the model find weights that work well on new data, not just on the training rows.

### 6.4 Why `shuffle=True`?

Without it, every epoch would cut the rows into exactly the same batches, in the same order. If the rows were ever sorted (for example all label-0 customers first), whole batches would hold only one class, and the weights would swing back and forth from batch to batch. Shuffling gives every batch a fair mix, different each epoch. Only the training data is shuffled. The validation and test sets are only used to measure the model, so their order does not matter.

## 7. Creating the Model

This is the same `CustomerPredictor` as in Part 1 ([`Notebook.ipynb`](Notebook.ipynb), Section 3): 3 inputs → 8 hidden neurons with ReLU → 1 raw score. Each notebook runs in its **own kernel**, so classes and variables from Part 1 do not exist here, and the class has to be defined again.

![CustomerPredictor network: 3 input features, a hidden layer of 8 neurons with ReLU, and 1 output](images/customer_predictor_network.svg)

- `torch.manual_seed(42)` right before `CustomerPredictor()` gives the model the same starting weights on every run.
- `print(model)` lists the two layers. The parameter count is 3 × 8 + 8 + 8 × 1 + 1 = **41**, the same as in Part 1.

In [8]:
# Create the class, same as in Part 1 (Notebook.ipynb).
# Each notebook runs in its own kernel, so the class must be defined again here.
class CustomerPredictor(nn.Module):
    def __init__(self):
        super().__init__()  # initialise nn.Module so the layers below get registered
        # input -> hidden: 3 input features, 8 neurons in one hidden layer
        self.hidden = nn.Linear(3, 8)
        # hidden -> output: 8 neurons in, 1 output value
        self.output = nn.Linear(8, 1)

    # forward pass: defines how an input flows through the network
    def forward(self, x):
        # Step 1: hidden layer applies z = x @ W.T + b -> shape (batch, 8)
        logits = self.hidden(x)
        # Step 2: ReLU keeps positive values and turns negative values into 0
        x = torch.relu(logits)
        # Step 3: output layer combines the 8 values into 1 raw score -> shape (batch, 1)
        return self.output(x)


# Fix the seed right before creating the model, so it starts with the same weights every run
torch.manual_seed(42)
model = CustomerPredictor()

print(model)  # the layers, in the order they were created
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))  # 41

CustomerPredictor(
  (hidden): Linear(in_features=3, out_features=8, bias=True)
  (output): Linear(in_features=8, out_features=1, bias=True)
)
Trainable parameters: 41


## 8. The Loss Function: Binary Cross-Entropy (BCE)

A **loss function** turns the model's mistakes into **one number**. Training then changes the weights to make that number smaller. Each customer's label is 0 or 1, so we use **binary cross-entropy**.

![Binary cross-entropy loss curves for true label 1 and true label 0](images/bce_loss_curve.svg)

| True label `y` | Predicted chance of 1, `p` | Loss | Meaning |
|---|---|---|---|
| 1 | 0.9 | 0.11 | Confident and right |
| 1 | 0.5 | 0.69 | A 50/50 guess |
| 1 | 0.1 | 2.30 | Confident and wrong |
| 0 | 0.1 | 0.11 | Confident and right |

**Why `nn.BCEWithLogitsLoss` and not `nn.BCELoss`?** The model returns a raw score (a logit), not a probability (Part 1, 3.5). `BCEWithLogitsLoss` applies the sigmoid itself and then the BCE, in one numerically stable step. `nn.BCELoss` would need a sigmoid inside the model first, and it can lose precision when the scores are very large or very small.

In [11]:
# BCE (Binary Cross Entropy) loss function: measures how far the model's predictions are from the true labels.
# BCEWithLogitsLoss does two steps in one:
#   1. sigmoid turns each raw score (logit) into a probability p between 0 and 1
#   2. BCE compares p with the true label y (0 or 1):
#        loss = -[y * log(p) + (1 - y) * log(1 - p)]
#      confident and right -> loss close to 0; confident and wrong -> large loss
# Combining them is more numerically stable than nn.Sigmoid() followed by nn.BCELoss(),
# which is why the model's output layer has no sigmoid.
loss_function = nn.BCEWithLogitsLoss()

# optim.AdamW is a popular optimiser that updates the model's weights to reduce the loss.
# model.parameters() gives the optimiser access to the model's weights, so it can update them.
# lr = 1e-3 is a common starting learning rate for AdamW, but it can be tuned.
# weight_decay=1e-4 is a common starting value for AdamW, but it can be tuned.
optimiser = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)